# EEG Preprocessing Pipeline

This notebook implements the project's foundational EEG preprocessing pipeline. Raw EDF recordings from three structurally distinct EEG domains — resting-state (REST), sleep polysomnography (SLEEP), and seizure recordings (CHB-MIT, subject chb01) — are loaded and processed independently through channel selection, band-pass filtering, fixed-duration epoching, spectral feature extraction, and stationarity diagnostics. Seizure annotations are parsed from the CHB-MIT summary file and used to construct an epoch-level seizure-overlap label. The resulting per-epoch feature tables, filtered epoch arrays, and preprocessing metadata are written to `outputs/` for use by downstream notebooks.

REST, SLEEP, and CHB are processed and stored as separate domains throughout the pipeline: epochs are never pooled or mixed across datasets, and any transformation fit on data (e.g. the feature scaler) is fit on a single, explicitly documented domain rather than assumed to generalize across domains.

In [ ]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import mne
import matplotlib
from pathlib import Path
from statsmodels.tsa.stattools import adfuller

print("=" * 60)
print("EEG PREPROCESSING PIPELINE V1")
print("=" * 60)
print("\nLibrary versions:")
print("  numpy:", np.__version__)
print("  pandas:", pd.__version__)
print("  scipy:", scipy.__version__)
print("  sklearn:", sklearn.__version__)
print("  mne:", mne.__version__)
print("  matplotlib:", matplotlib.__version__)

## Data Discovery and Preprocessing Configuration

File paths for the REST, SLEEP, and CHB-MIT source recordings are declared and validated against the filesystem before any processing begins, so a missing input fails immediately and explicitly rather than downstream.

The preprocessing parameters used throughout the notebook are fixed here: a 0.5–40 Hz band-pass filter, 30-second non-overlapping epochs, and a significance level of 0.05 for the stationarity test applied later. The zero-overlap epoching convention keeps epoch-index arithmetic exactly equal to elapsed time, which the transition-horizon semantics used in downstream forecasting notebooks depend on.

In [ ]:
print("\n" + "=" * 60)
print("STEP 1: DISCOVER AND VALIDATE DATA PATHS")
print("=" * 60)

data_root = Path("/home/evgenija/projects/EEG_timeseries_project/data")

# Resting state EEG (REST dataset)
rest_dir = data_root / "raw" / "rest"
rest_files = {
    "rest_s001_r01": rest_dir / "S001R01.edf",
    "rest_s001_r02": rest_dir / "S001R02.edf",
}

# Sleep EEG (SLEEP dataset)
sleep_dir = data_root / "raw" / "sleep"
sleep_files = {
    "sleep_sc4001": sleep_dir / "SC4001E0-PSG.edf",
}

# Seizure EEG (CHB-MIT dataset)
chb_dir = data_root / "raw" / "chbmit"
chb_files = {
    "chb_s01_r01": chb_dir / "chb01_01.edf",
    "chb_s01_r02": chb_dir / "chb01_02.edf",
    "chb_s01_r03": chb_dir / "chb01_03.edf",
}

# Validate all files exist
all_files = {**rest_files, **sleep_files, **chb_files}
for name, path in all_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Expected file not found: {path}")
    print(f"  ✓ {name:20s} {path.stat().st_size / 1e6:8.1f} MB")

print(f"\nTotal files found: {len(all_files)}")
print(f"REST: {len(rest_files)}, SLEEP: {len(sleep_files)}, CHB: {len(chb_files)}")

In [ ]:
print("\n" + "=" * 60)
print("STEP 2: PREPROCESSING CONFIGURATION")
print("=" * 60)

# Preprocessing parameters (established project decisions, CLAUDE.md)
FILTER_LOW_HZ = 0.5
FILTER_HIGH_HZ = 40.0

# Epoching parameters (established temporal convention, CLAUDE.md)
EPOCH_DURATION_SEC = 30.0
EPOCH_OVERLAP = 0.0

# Stationarity check parameters
STATIONARITY_ADF_ALPHA = 0.05

print(f"\nFiltering: {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz")
print(f"Epochs: {EPOCH_DURATION_SEC}s, overlap={EPOCH_OVERLAP}")
print(f"Stationarity check: ADF alpha={STATIONARITY_ADF_ALPHA}")

# Output directories
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

data_dir = output_dir / "preprocessed_data"
data_dir.mkdir(exist_ok=True)

print(f"\nOutput directory: {output_dir.resolve()}")
print(f"Data directory: {data_dir.resolve()}")

## Raw Recording Loading

Each EDF recording is loaded with MNE and its channel count, sampling frequency, and duration are reported. Sampling frequency is verified to be consistent across the two REST recordings and across the three CHB recordings; REST, SLEEP, and CHB are otherwise not required to share a sampling frequency, since they are separate acquisition systems.

In [ ]:
print("\n" + "=" * 60)
print("STEP 3: LOAD AND INSPECT RAW DATA")
print("=" * 60)

def load_edf_validated(path, dataset_name):
    """Load EDF and validate basic structure."""
    print(f"\nLoading {dataset_name}: {path.name}")
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    print(f"  Channels: {len(raw.ch_names)}")
    print(f"  Sfreq: {raw.info['sfreq']} Hz")
    print(f"  Duration: {raw.times[-1]:.1f}s ({raw.times[-1]/3600:.2f}h)")
    return raw

# Load REST dataset
rest_r01 = load_edf_validated(rest_files["rest_s001_r01"], "REST - Subject 001 Recording 01")
rest_r02 = load_edf_validated(rest_files["rest_s001_r02"], "REST - Subject 001 Recording 02")

sfreq_rest = float(rest_r01.info["sfreq"])
if not np.isclose(sfreq_rest, rest_r02.info["sfreq"]):
    raise ValueError("REST recordings have different sampling rates!")

# Load SLEEP dataset
sleep = load_edf_validated(sleep_files["sleep_sc4001"], "SLEEP - SC4001E0")
sfreq_sleep = float(sleep.info["sfreq"])
print(f"  Sleep channels (raw): {sleep.ch_names}")

# Load CHB-MIT dataset (seizure recordings)
chb_recordings = {}
for rec_name, file_path in chb_files.items():
    raw = load_edf_validated(file_path, f"CHB-MIT - {rec_name}")
    chb_recordings[rec_name] = raw

sfreq_chb = float(chb_recordings["chb_s01_r01"].info["sfreq"])
for name, raw in chb_recordings.items():
    if not np.isclose(raw.info["sfreq"], sfreq_chb):
        raise ValueError("CHB recordings have different sampling rates!")

print(f"\n✓ All raw data loaded")
print(f"  REST sfreq: {sfreq_rest} Hz")
print(f"  SLEEP sfreq: {sfreq_sleep} Hz")
print(f"  CHB sfreq: {sfreq_chb} Hz")

## Channel Type Correction and Harmonization

The SLEEP polysomnography recording contains multiple physiological signal types (EEG, EOG, EMG, respiration, temperature, and an event marker) that MNE's EDF reader initially types uniformly as EEG. Channel types are corrected explicitly from the known channel names in this file, so that a subsequent EEG-only channel selection does not inadvertently include non-EEG signals such as EOG or EMG.

REST, SLEEP, and CHB use structurally different electrode montages (a 64-channel montage for REST, a 2-channel montage for SLEEP, and a clinical bipolar montage for CHB). No channel is invented, interpolated, or mapped across datasets to force a common channel set; each domain is processed with its own native channels, and any cross-dataset channel correspondence is left as a modeling-stage decision rather than being resolved during preprocessing.

In [ ]:
print("\n" + "=" * 60)
print("STEP 4: SLEEP CHANNEL TYPE CORRECTION")
print("=" * 60)

# The Sleep PSG (SC4001E0) contains multiple physiological signal types.
# MNE's EDF reader initially types all 7 channels as EEG, which would
# incorrectly let a naive pick(picks="eeg") select EOG/EMG/respiration/
# temperature/stim channels alongside genuine EEG. We correct channel
# types explicitly, based on the actual channel names in the file.
sleep_channel_types = {
    "EEG Fpz-Cz": "eeg",
    "EEG Pz-Oz": "eeg",
    "EOG horizontal": "eog",
    "Resp oro-nasal": "resp",
    "EMG submental": "emg",
    "Temp rectal": "misc",
    "Event marker": "stim",
}

missing_channels = [ch for ch in sleep_channel_types if ch not in sleep.ch_names]
if missing_channels:
    raise ValueError(f"Sleep: Missing expected channels: {missing_channels}")

sleep.set_channel_types(sleep_channel_types, verbose=False)

print("\nCorrected channel types:")
for ch, ch_type in zip(sleep.ch_names, sleep.get_channel_types()):
    print(f"  {ch:20s} -> {ch_type}")

sleep_eeg_only = [ch for ch, t in zip(sleep.ch_names, sleep.get_channel_types()) if t == "eeg"]
expected_sleep_eeg = ["EEG Fpz-Cz", "EEG Pz-Oz"]

if sleep_eeg_only != expected_sleep_eeg:
    raise ValueError(
        f"Sleep EEG channel selection mismatch. "
        f"Expected {expected_sleep_eeg}, got {sleep_eeg_only}"
    )

print(f"\n✓ Verified EEG-only channels: {sleep_eeg_only}")

In [ ]:
print("\n" + "=" * 60)
print("STEP 5: CHANNEL INSPECTION & HARMONIZATION")
print("=" * 60)

rest_eeg_channels = [ch for ch in rest_r01.ch_names if rest_r01.get_channel_types([ch])[0] == "eeg"]
chb_eeg_channels = [ch for ch in chb_recordings["chb_s01_r01"].ch_names
                    if chb_recordings["chb_s01_r01"].get_channel_types([ch])[0] == "eeg"]

print("\nREST Dataset:")
print(f"  EEG channels: {len(rest_eeg_channels)} (64-electrode montage)")

print("\nSLEEP Dataset:")
print(f"  EEG channels: {sleep_eeg_only}")

print("\nCHB-MIT Dataset:")
print(f"  EEG channels: {len(chb_eeg_channels)} (clinical bipolar montage)")

print("\nHarmonization notes:")
print("  - REST, SLEEP, and CHB use structurally different montages.")
print("  - No channel is invented or interpolated across datasets.")
print("  - Each dataset is processed with its own native channel set.")
print("  - Cross-dataset channel mapping (if ever needed) is a modelling-")
print("    stage decision, not a preprocessing-stage one.")

## EEG Channel Selection and Band-Pass Filtering

For each recording, only EEG-typed channels are retained and a 0.5–40 Hz band-pass filter is applied. The requested high cutoff is checked against the recording's Nyquist frequency before filtering, and the resulting signal is validated to be finite. This selection-then-filter-then-validate order is applied uniformly across REST, SLEEP, and CHB.

In [ ]:
print("\n" + "=" * 60)
print("STEP 6: PREPROCESSING (EEG SELECTION, FILTERING, VALIDATION)")
print("=" * 60)

def preprocess_eeg(raw, dataset_name, l_freq=FILTER_LOW_HZ, h_freq=FILTER_HIGH_HZ):
    """
    Select EEG-typed channels, apply band-pass filter, validate.

    Order: correct channel types (done upstream for Sleep) -> select EEG
    -> filter -> validate -> convert to NumPy. This avoids converting to
    NumPy before MNE-level corrections are applied.

    Returns:
        eeg_raw: MNE Raw object (EEG channels only, filtered)
        data: NumPy array (channels, samples)
        sfreq: Sampling frequency
    """
    eeg_raw = raw.copy().pick(picks="eeg")

    if len(eeg_raw.ch_names) == 0:
        raise ValueError(f"{dataset_name}: No EEG channels found")

    sfreq = float(eeg_raw.info["sfreq"])
    nyquist = sfreq / 2.0

    if h_freq >= nyquist:
        raise ValueError(
            f"{dataset_name}: filter cutoff {h_freq}Hz >= Nyquist {nyquist}Hz "
            f"(sfreq={sfreq}Hz)"
        )

    eeg_raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)

    data = eeg_raw.get_data()

    if data.ndim != 2:
        raise ValueError(f"{dataset_name}: Expected 2D data, got {data.shape}")

    if not np.isfinite(data).all():
        raise ValueError(f"{dataset_name}: Data contains NaN/Inf after filtering")

    print(f"{dataset_name:16s}: channels={len(eeg_raw.ch_names):2d}, "
          f"sfreq={sfreq:6.1f}Hz, shape={data.shape}, "
          f"filter=[{l_freq:.1f}, {h_freq:.1f}]Hz")

    return eeg_raw, data, sfreq

print("\nREST Dataset:")
rest_r01_eeg, rest_r01_data, _ = preprocess_eeg(rest_r01, "rest_s001_r01")
rest_r02_eeg, rest_r02_data, _ = preprocess_eeg(rest_r02, "rest_s001_r02")

print("\nSLEEP Dataset:")
sleep_eeg, sleep_data, _ = preprocess_eeg(sleep, "sleep_sc4001")

if sleep_eeg.ch_names != expected_sleep_eeg:
    raise ValueError(
        f"Sleep EEG selection mismatch after filtering. "
        f"Expected {expected_sleep_eeg}, got {sleep_eeg.ch_names}"
    )

print("\nCHB-MIT Dataset:")
chb_data_dict = {}
for rec_name, raw in chb_recordings.items():
    raw_eeg, data, _ = preprocess_eeg(raw, rec_name)
    chb_data_dict[rec_name] = (raw_eeg, data)

print("\n✓ EEG selection, filtering, and validation complete for all datasets")

## Temporal Window (Epoch) Construction

Each filtered recording is divided into fixed-duration, non-overlapping 30-second epochs. Windows are constructed independently per recording, and any samples remaining after the last full window are discarded rather than zero-padded, so that every epoch reflects an equal amount of real signal. Zero overlap preserves a direct correspondence between epoch index and elapsed time, which later notebooks rely on when constructing time-to-event forecasting targets.

In [ ]:
print("\n" + "=" * 60)
print("STEP 7: EPOCHING")
print("=" * 60)

def make_epochs(data, sfreq, win_sec=EPOCH_DURATION_SEC, overlap=EPOCH_OVERLAP):
    """
    Create fixed-duration, non-overlapping epochs from continuous data.

    Temporal convention (established, CLAUDE.md):
        +1 epoch = 30 s, +2 epochs = 60 s, +4 epochs = 120 s.
    Zero overlap is deliberate — it keeps epoch-index arithmetic exactly
    equal to elapsed time, which downstream transition-horizon semantics
    (30s/60s/120s) depend on. Leftover samples that don't fill a full
    window are dropped (not zero-padded), so all epochs have equal, real
    signal content.

    Parameters:
        data: (channels, samples)
        sfreq: Sampling frequency in Hz
        win_sec: Window duration in seconds
        overlap: Overlap fraction [0, 1)

    Returns:
        epochs: (n_epochs, channels, win_samples)
    """
    if data.ndim != 2:
        raise ValueError(f"Expected 2D data, got {data.shape}")

    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0, 1)")

    win_samp = int(round(win_sec * sfreq))
    step = int(round(win_samp * (1 - overlap)))

    if win_samp <= 0:
        raise ValueError("win_sec and sfreq must produce positive window size")

    if step <= 0:
        raise ValueError("overlap is too large")

    n_samples = data.shape[1]

    if n_samples < win_samp:
        raise ValueError(
            f"Recording has {n_samples} samples, but epoch requires {win_samp}"
        )

    starts = np.arange(0, n_samples - win_samp + 1, step)

    epochs = np.stack(
        [data[:, start:start + win_samp] for start in starts],
        axis=0
    )

    return epochs

def validate_epochs(name, epochs, sfreq, win_sec=EPOCH_DURATION_SEC,
                   expected_channels=None):
    """Validate epoch structure and content (shape, channels, finiteness)."""
    expected_samples = int(round(win_sec * sfreq))

    if epochs.ndim != 3:
        raise ValueError(f"{name}: Expected 3D, got {epochs.shape}")

    if epochs.shape[2] != expected_samples:
        raise ValueError(
            f"{name}: Expected {expected_samples} samples/epoch, got {epochs.shape[2]}"
        )

    if expected_channels is not None:
        if epochs.shape[1] != expected_channels:
            raise ValueError(
                f"{name}: Expected {expected_channels} channels, got {epochs.shape[1]}"
            )

    if not np.isfinite(epochs).all():
        raise ValueError(f"{name}: Contains NaN/Inf values")

    print(f"{name:16s}: epochs={epochs.shape[0]:5d}, channels={epochs.shape[1]:2d}, "
          f"samples={epochs.shape[2]:5d}, duration={win_sec:.0f}s")

print("\nREST Dataset:")
ep_rest_r01 = make_epochs(rest_r01_data, sfreq_rest)
ep_rest_r02 = make_epochs(rest_r02_data, sfreq_rest)
validate_epochs("rest_s001_r01", ep_rest_r01, sfreq_rest)
validate_epochs("rest_s001_r02", ep_rest_r02, sfreq_rest)

print("\nSLEEP Dataset:")
ep_sleep = make_epochs(sleep_data, sfreq_sleep)
validate_epochs("sleep_sc4001", ep_sleep, sfreq_sleep, expected_channels=2)

print("\nCHB-MIT Dataset:")
ep_chb_dict = {}
for rec_name, (_, data) in chb_data_dict.items():
    epochs = make_epochs(data, sfreq_chb)
    ep_chb_dict[rec_name] = epochs
    validate_epochs(rec_name, epochs, sfreq_chb)

print("\n✓ Epoching complete — chronological order preserved within each recording")

## Spectral and Statistical Feature Extraction

For each epoch, Welch's method is used to estimate the power spectral density per channel, from which absolute power in the delta (0.5–4 Hz), theta (4–8 Hz), alpha (8–13 Hz), and beta (13–30 Hz) bands is computed and averaged across channels. Relative band powers are obtained by normalizing each absolute band power by their sum, and two clinically motivated band ratios (theta/alpha, delta/beta) are computed. A normalized spectral entropy is derived from the same Welch PSD estimate. Alongside these spectral features, basic amplitude statistics (standard deviation, variance, RMS) are computed directly from the raw epoch signal. Each recording's `recording_id` is attached to its feature rows, providing the identifier used for dataset- and recording-level separation in later stages.

In [ ]:
print("\n" + "=" * 60)
print("STEP 8: FEATURE EXTRACTION (WELCH PSD, BAND POWERS)")
print("=" * 60)

from scipy.signal import welch

def bandpower(x, sfreq, fmin, fmax, nperseg=None):
    """Absolute band power in [fmin, fmax] via Welch PSD (trapezoidal integration)."""
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))  # ~2s sub-windows for PSD estimation
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return np.nan
    return np.trapezoid(psd[mask], freqs[mask])

def spectral_entropy_welch(x, sfreq, fmin=1.0, fmax=40.0, nperseg=None, eps=1e-12):
    """Shannon entropy of the normalized Welch PSD, normalized to [0, 1]."""
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    psd_band = np.maximum(psd[mask], eps)
    p = psd_band / psd_band.sum()
    ent = -np.sum(p * np.log(p))
    return ent / np.log(len(p))

def extract_features(epochs, sfreq):
    """
    Extract qEEG spectral features from epochs, averaged across channels.

    Bands: delta 0.5-4Hz, theta 4-8Hz, alpha 8-13Hz, beta 13-30Hz.
    """
    features = []
    eps = 1e-12

    for ep in epochs:
        std = np.std(ep)
        var = np.var(ep)
        rms = np.sqrt(np.mean(ep**2))

        delta = np.mean([bandpower(ch, sfreq, 0.5, 4) for ch in ep])
        theta = np.mean([bandpower(ch, sfreq, 4, 8) for ch in ep])
        alpha = np.mean([bandpower(ch, sfreq, 8, 13) for ch in ep])
        beta = np.mean([bandpower(ch, sfreq, 13, 30) for ch in ep])

        total_power = delta + theta + alpha + beta + eps
        rel_delta = delta / total_power
        rel_theta = theta / total_power
        rel_alpha = alpha / total_power
        rel_beta = beta / total_power

        theta_alpha_ratio = theta / (alpha + eps)
        delta_beta_ratio = delta / (beta + eps)

        ent = np.mean([spectral_entropy_welch(ch, sfreq) for ch in ep])

        features.append({
            "std": std, "var": var, "rms": rms,
            "delta": delta, "theta": theta, "alpha": alpha, "beta": beta,
            "rel_delta": rel_delta, "rel_theta": rel_theta,
            "rel_alpha": rel_alpha, "rel_beta": rel_beta,
            "theta_alpha_ratio": theta_alpha_ratio,
            "delta_beta_ratio": delta_beta_ratio,
            "entropy": ent,
        })

    return pd.DataFrame(features)

print("\nExtracting features...")
feat_rest_r01 = extract_features(ep_rest_r01, sfreq_rest)
feat_rest_r02 = extract_features(ep_rest_r02, sfreq_rest)
feat_sleep = extract_features(ep_sleep, sfreq_sleep)

feat_chb_dict = {}
for rec_name, epochs in ep_chb_dict.items():
    feat_chb_dict[rec_name] = extract_features(epochs, sfreq_chb)

# Recording identity (required for cross-dataset evaluation / leave-one-out)
feat_rest_r01["recording_id"] = "rest_s001_r01"
feat_rest_r02["recording_id"] = "rest_s001_r02"
feat_sleep["recording_id"] = "sleep_sc4001"
for rec_name, feat in feat_chb_dict.items():
    feat["recording_id"] = rec_name

feat_rest_all = pd.concat([feat_rest_r01, feat_rest_r02], ignore_index=True)
feat_chb_all = pd.concat(list(feat_chb_dict.values()), ignore_index=True)

print(f"  REST:  {feat_rest_all.shape[0]} epochs x {feat_rest_all.shape[1]} cols")
print(f"  SLEEP: {feat_sleep.shape[0]} epochs x {feat_sleep.shape[1]} cols")
print(f"  CHB:   {feat_chb_all.shape[0]} epochs x {feat_chb_all.shape[1]} cols")
print("\n✓ Feature extraction complete")

## Stationarity Assessment

Stationarity is assessed per recording using two complementary diagnostics on a single representative channel: rolling mean and rolling standard deviation over a 5-second window, and the Augmented Dickey-Fuller (ADF) test applied to a uniformly spaced sample of epochs, reporting the fraction of sampled epochs for which the null hypothesis of a unit root is rejected at alpha=0.05. Datasets are evaluated separately rather than on a concatenated signal, since pooling REST, SLEEP, and CHB would let domain differences dominate the resulting statistic. This step is diagnostic only: no differencing, detrending, or other stationarity-forcing transform is applied to the data based on its result.

In [ ]:
print("\n" + "=" * 60)
print("STEP 9: STATIONARITY ANALYSIS (rolling mean/std + ADF test)")
print("=" * 60)

# Required diagnostics (established, project-state.md 5.7): rolling mean,
# rolling standard deviation, and the Augmented Dickey-Fuller test, run
# separately per dataset/recording — never on a concatenated REST+SLEEP+CHB
# signal, since domain differences would dominate any such statistic.
#
# A non-stationary result does NOT by itself authorize differencing,
# detrending, or forced-stationarity transforms; any such transform would
# need separate scientific justification tied to the forecasting target.

def rolling_stats(x, sfreq, window_sec=5.0):
    """Rolling mean and rolling std of a 1D signal using a window_sec window."""
    win = max(1, int(round(window_sec * sfreq)))
    s = pd.Series(x)
    return s.rolling(win).mean().to_numpy(), s.rolling(win).std().to_numpy()

def adf_test_epochs(epochs, sfreq, dataset_name, sample_size=10):
    """
    ADF test on a uniformly-spaced sample of epochs (single representative
    channel: channel 0), reporting the fraction of epochs where the null
    hypothesis of a unit root is rejected (p < STATIONARITY_ADF_ALPHA).
    """
    n_epochs = epochs.shape[0]
    sample_indices = np.linspace(0, n_epochs - 1, min(sample_size, n_epochs), dtype=int)

    rejections = 0
    for idx in sample_indices:
        ch0 = epochs[idx, 0, :]
        result = adfuller(ch0, autolag="AIC")
        if result[1] < STATIONARITY_ADF_ALPHA:
            rejections += 1

    rejection_rate = rejections / len(sample_indices)

    # Rolling mean/std on one representative epoch (first sampled) to
    # visualize within-epoch stability, reported as a summary statistic.
    example_epoch = epochs[sample_indices[0], 0, :]
    roll_mean, roll_std = rolling_stats(example_epoch, sfreq)
    roll_mean_range = np.nanmax(roll_mean) - np.nanmin(roll_mean)
    roll_std_range = np.nanmax(roll_std) - np.nanmin(roll_std)

    print(f"{dataset_name:16s}: ADF rejection_rate={rejection_rate:.1%} "
          f"({rejections}/{len(sample_indices)} epochs, ch0), "
          f"rolling_mean_range={roll_mean_range:.3g}, "
          f"rolling_std_range={roll_std_range:.3g}")

    return rejection_rate

print("\nTesting stationarity per recording (channel 0, sampled epochs)...")
print("\nREST:")
adf_test_epochs(ep_rest_r01, sfreq_rest, "rest_s001_r01")
adf_test_epochs(ep_rest_r02, sfreq_rest, "rest_s001_r02")

print("\nSLEEP:")
adf_test_epochs(ep_sleep, sfreq_sleep, "sleep_sc4001")

print("\nCHB-MIT:")
for rec_name, epochs in ep_chb_dict.items():
    adf_test_epochs(epochs, sfreq_chb, rec_name)

print("\nInterpretation:")
print("  - ADF rejection_rate: fraction of sampled epochs classified stationary (p<0.05).")
print("  - rolling_mean/std_range: within-epoch drift on one representative epoch.")
print("  - EEG is expected to be quasi-stationary at the 30s scale; large deviations")
print("    across datasets reflect known domain differences (REST/SLEEP/CHB), not")
print("    necessarily evidence about any single physiological state.")
print("  - No transform is applied here — see note above.")

## Temporal Indexing

A per-recording epoch index and corresponding elapsed-time columns (`t_start_sec`, `t_center_sec`, `t_end_sec`) are added to each feature table. The epoch index restarts at zero for every recording, so elapsed time is always relative to that recording's own start and never crosses a recording or dataset boundary. This temporal index is a time axis only; it does not encode a physiological state or deviation measure, and no such state-index or transition target is constructed in this notebook.

In [ ]:
print("\n" + "=" * 60)
print("STEP 10: TEMPORAL METADATA (forecasting index)")
print("=" * 60)

def add_temporal_metadata(df, win_sec=EPOCH_DURATION_SEC):
    """
    Add per-recording epoch index and elapsed-time columns.

    epoch_idx restarts at 0 for every recording (groupby recording_id) so
    that time is always relative to that recording's own start — this is
    the forecasting index the project currently supports. It intentionally
    does NOT cross recording or dataset boundaries, and it does not encode
    any state/transition target (no NSDI / state-geometry index is
    computed here — see markdown note below).
    """
    df = df.copy()

    if "recording_id" not in df.columns:
        raise ValueError("Expected 'recording_id' column")

    df["epoch_idx"] = df.groupby("recording_id").cumcount()
    df["t_start_sec"] = df["epoch_idx"] * win_sec
    df["t_center_sec"] = df["t_start_sec"] + win_sec / 2.0
    df["t_end_sec"] = df["t_start_sec"] + win_sec

    return df

feat_rest_all = add_temporal_metadata(feat_rest_all)
feat_sleep = add_temporal_metadata(feat_sleep)
feat_chb_all = add_temporal_metadata(feat_chb_all)

print("\nEpochs per recording:")
print(pd.concat([
    feat_rest_all[["recording_id"]],
    feat_sleep[["recording_id"]],
    feat_chb_all[["recording_id"]],
]).groupby("recording_id").size())

print("\n✓ Temporal metadata added (epoch_idx, t_start_sec, t_center_sec, t_end_sec)")

## CHB-MIT Seizure Annotation Parsing and Epoch Labeling

Seizure onset and termination times for the CHB-MIT recordings are parsed directly from the subject's official summary file (`chb01-summary.txt`), restricted to the recordings actually loaded in this notebook, and cross-checked against the declared per-file seizure count so a parsing error fails loudly rather than silently mislabeling epochs. An epoch is labeled `y_seizure = 1` when its interval `[t_start_sec, t_end_sec)` intersects an annotated seizure interval at all (interval intersection, not epoch-center membership), so that an epoch containing a seizure onset or termination is still marked positive.

REST and SLEEP have no seizure annotation source and therefore receive no `y_seizure` column at all, rather than a column filled with negative labels; the absence of an annotation source is kept structurally distinct from a confirmed negative label.

In [ ]:
print("\n" + "=" * 60)
print("STEP 10B: CHB SEIZURE EPOCH-LEVEL LABELING")
print("=" * 60)

import re

# Provenance: seizure intervals are parsed directly from the official
# CHB-MIT summary file for this subject (data/raw/chbmit/chb01-summary.txt),
# not hardcoded blind. This is the only annotation source this project has
# for CHB. Parsing is restricted to recordings this notebook actually
# loaded, so an unrelated block elsewhere in the file cannot silently
# inject an annotation for a recording we never touched.
#
# REST and SLEEP have no seizure annotation source at all, so they
# deliberately receive NO y_seizure column (not a column of 0s) — a
# missing column is structurally distinct from a confirmed negative label,
# per the project's constraint against fabricating labels.

def parse_chb_summary(summary_path, known_edf_filenames):
    """
    Parse a CHB-MIT *-summary.txt file into {edf_filename: [(start_s, end_s), ...]}.

    Only entries whose File Name is in `known_edf_filenames` are kept.
    Raises if a file's declared seizure count doesn't match the number of
    Start/End time pairs actually parsed for it, or if any known
    recording has no summary entry at all.
    """
    text = summary_path.read_text()
    blocks = re.split(r"(?=File Name: )", text)

    annotations = {}
    for block in blocks:
        name_match = re.search(r"File Name:\s*(\S+)", block)
        if not name_match:
            continue
        fname = name_match.group(1)
        if fname not in known_edf_filenames:
            continue

        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not n_match:
            raise ValueError(f"{fname}: could not find seizure count in summary")
        n_seizures = int(n_match.group(1))

        starts = [int(s) for s in re.findall(r"Seizure Start Time:\s*(\d+)\s*seconds", block)]
        ends = [int(s) for s in re.findall(r"Seizure End Time:\s*(\d+)\s*seconds", block)]

        if len(starts) != n_seizures or len(ends) != n_seizures:
            raise ValueError(
                f"{fname}: declared {n_seizures} seizures but parsed "
                f"{len(starts)} start / {len(ends)} end times"
            )

        annotations[fname] = list(zip(starts, ends))

    missing = set(known_edf_filenames) - set(annotations)
    if missing:
        raise ValueError(f"No summary entry found for: {missing}")

    return annotations

chb_edf_filenames = {rec_name: chb_files[rec_name].name for rec_name in chb_files}
chb_summary_path = chb_dir / "chb01-summary.txt"

chb_seizure_intervals_by_file = parse_chb_summary(
    chb_summary_path,
    known_edf_filenames=set(chb_edf_filenames.values()),
)
chb_seizure_intervals = {
    rec_name: chb_seizure_intervals_by_file[fname]
    for rec_name, fname in chb_edf_filenames.items()
}

print(f"\nParsed from {chb_summary_path.name}:")
for rec_name, intervals in chb_seizure_intervals.items():
    if intervals:
        for start, end in intervals:
            print(f"  {rec_name}: seizure {start}-{end}s")
    else:
        print(f"  {rec_name}: no seizures")

# Sanity-check against the known ground truth for this subject, so a
# parsing regression fails loudly rather than silently mislabeling epochs.
expected_chb_intervals = {
    "chb_s01_r01": [],
    "chb_s01_r02": [],
    "chb_s01_r03": [(2996, 3036)],
}
if chb_seizure_intervals != expected_chb_intervals:
    raise ValueError(
        f"Parsed CHB seizure intervals {chb_seizure_intervals} do not match "
        f"expected {expected_chb_intervals}"
    )

def label_seizure_epochs(t_start_sec, t_end_sec, intervals):
    """
    y_seizure = 1 iff [t_start_sec, t_end_sec) intersects any annotated
    seizure interval, using true interval intersection:
        epoch_start < seizure_end AND epoch_end > seizure_start
    (not epoch-center membership) — a seizure can begin or end inside a
    30s epoch and that epoch should still be marked positive.
    """
    y = np.zeros(len(t_start_sec), dtype=int)
    for i, (t_start, t_end) in enumerate(zip(t_start_sec, t_end_sec)):
        for seiz_start, seiz_end in intervals:
            if t_start < seiz_end and t_end > seiz_start:
                y[i] = 1
                break
    return y

feat_chb_all["y_seizure"] = 0
for rec_name, intervals in chb_seizure_intervals.items():
    mask = feat_chb_all["recording_id"] == rec_name
    feat_chb_all.loc[mask, "y_seizure"] = label_seizure_epochs(
        feat_chb_all.loc[mask, "t_start_sec"],
        feat_chb_all.loc[mask, "t_end_sec"],
        intervals,
    )

n_seizure_positive = int(feat_chb_all["y_seizure"].sum())
n_seizure_negative = len(feat_chb_all) - n_seizure_positive

assert set(feat_chb_all["y_seizure"].unique()) <= {0, 1}, "y_seizure must be binary {0,1}"

print(f"\n✓ y_seizure labeled: {n_seizure_positive} positive epochs, {n_seizure_negative} negative epochs")
print("\nPositive epochs:")
print(
    feat_chb_all.loc[
        feat_chb_all["y_seizure"] == 1,
        ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec"]
    ].to_string(index=False)
)

print("\nSLEEP and REST receive no y_seizure column (no annotation source for either).")
assert "y_seizure" not in feat_sleep.columns
assert "y_seizure" not in feat_rest_all.columns

## Feature Scaling

A `RobustScaler` (median / interquartile-range based) is fit on the REST feature set and applied, unchanged, to REST, SLEEP, and CHB. A robust scaler is used in preference to a mean/variance scaler for resilience to outlier epochs, such as the large-amplitude segments present during a CHB seizure. Fitting on REST — rather than on SLEEP or CHB — keeps the scaling domain-neutral with respect to the two datasets it is subsequently applied to, at the cost of a fragile median/IQR estimate given that REST currently contains only 4 epochs; this limitation is carried forward explicitly rather than resolved by fitting on a larger but methodologically less appropriate domain.

In [ ]:
print("\n" + "=" * 60)
print("STEP 11: FEATURE SCALING")
print("=" * 60)

from sklearn.preprocessing import RobustScaler

# RobustScaler (median/IQR) chosen over StandardScaler for resilience to
# EEG outlier epochs (e.g. seizure-range amplitudes in CHB).
# Fit on REST only: REST is the resting baseline domain, and fitting on
# CHB or SLEEP would let those domains' own extreme values leak into the
# scale used to compare all datasets. This is a per-dataset-application,
# single-fit design consistent with not mixing domains during preprocessing.
#
# KNOWN LIMITATION (documented, not fixed here): REST currently has only
# 4 epochs, so the fitted median/IQR are fragile estimates of the resting
# baseline. Fitting on SLEEP or CHB instead would avoid the small-N issue
# but would leak that domain's own distribution into the shared scale
# (SLEEP/CHB are exactly the domains this scaler is later applied to), and
# fitting jointly across domains would violate the project's domain-
# separation constraint. No alternative here is obviously safer without
# either more REST recordings or a modeling-stage decision (e.g. per-
# domain scaling revisited at train time) — so the current REST-only fit
# is kept, and this limitation is carried forward rather than silently
# resolved. See project-state.md for the broader domain-separation rule.
feature_cols_to_scale = ["std", "var", "rms", "delta", "theta", "alpha", "beta", "entropy"]

scaler = RobustScaler()
scaler.fit(feat_rest_all[feature_cols_to_scale])

for df in (feat_rest_all, feat_sleep, feat_chb_all):
    scaled = scaler.transform(df[feature_cols_to_scale])
    for i, col in enumerate(feature_cols_to_scale):
        df[f"{col}_scaled"] = scaled[:, i]

print(f"Scaler fit on: REST ({len(feat_rest_all)} epochs) — KNOWN LIMITATION: small N, see comment above")
print(f"Applied to: REST, SLEEP ({len(feat_sleep)} epochs), CHB ({len(feat_chb_all)} epochs)")
print("\n✓ Scaling complete")

## Output Artifact Export

Per-epoch feature tables (CSV and Parquet) are written for each of the three domains, along with the corresponding filtered epoch arrays (NumPy `.npy`, pre-feature-extraction) and a `preprocessing_metadata.json` file recording the preprocessing parameters, sampling frequencies, recording identifiers, epoch counts, seizure-annotation provenance, and known limitations (in particular, the small REST sample size underlying the fitted scaler).

In [ ]:
print("\n" + "=" * 60)
print("STEP 12: SAVE PREPROCESSED OUTPUTS")
print("=" * 60)

feat_rest_all.to_csv(output_dir / "features_rest.csv", index=False)
feat_sleep.to_csv(output_dir / "features_sleep.csv", index=False)
feat_chb_all.to_csv(output_dir / "features_chb.csv", index=False)

feat_rest_all.to_parquet(output_dir / "features_rest.parquet", index=False)
feat_sleep.to_parquet(output_dir / "features_sleep.parquet", index=False)
feat_chb_all.to_parquet(output_dir / "features_chb.parquet", index=False)

print("Feature tables saved (CSV + Parquet):")
print(f"  features_rest  : {feat_rest_all.shape}")
print(f"  features_sleep : {feat_sleep.shape}")
print(f"  features_chb   : {feat_chb_all.shape}")

np.save(data_dir / "epochs_rest_r01.npy", ep_rest_r01)
np.save(data_dir / "epochs_rest_r02.npy", ep_rest_r02)
np.save(data_dir / "epochs_sleep.npy", ep_sleep)
for rec_name, epochs in ep_chb_dict.items():
    np.save(data_dir / f"epochs_{rec_name}.npy", epochs)

print(f"\nEpoch arrays (filtered EEG, pre-feature-extraction) saved to {data_dir}/")

import json

metadata = {
    "preprocessing_params": {
        "filter_low_hz": FILTER_LOW_HZ,
        "filter_high_hz": FILTER_HIGH_HZ,
        "epoch_duration_sec": EPOCH_DURATION_SEC,
        "epoch_overlap": EPOCH_OVERLAP,
    },
    "sampling_rates_hz": {
        "rest": sfreq_rest,
        "sleep": sfreq_sleep,
        "chb": sfreq_chb,
    },
    "recordings": {
        "rest": list(rest_files.keys()),
        "sleep": list(sleep_files.keys()),
        "chb": list(chb_files.keys()),
    },
    "n_epochs": {
        "rest": int(len(feat_rest_all)),
        "sleep": int(len(feat_sleep)),
        "chb": int(len(feat_chb_all)),
    },
    "supervised_targets": {
        "y_sleep": "NOT AVAILABLE - no verified sleep-stage hypnogram in this repository",
        "y_seizure": {
            "status": "IMPLEMENTED",
            "applies_to": "chb only (no column on rest/sleep - absence is structural, not a negative label)",
            "annotation_source": str(chb_summary_path),
            "seizure_intervals_sec": {
                rec_name: intervals for rec_name, intervals in chb_seizure_intervals.items()
            },
            "labeling_policy": (
                "epoch marked 1 iff [t_start_sec, t_end_sec) intersects an "
                "annotated interval: epoch_start < seizure_end AND "
                "epoch_end > seizure_start (interval intersection, not "
                "epoch-center membership)"
            ),
            "n_positive_epochs": n_seizure_positive,
            "n_negative_epochs": n_seizure_negative,
        },
    },
    "known_limitations": {
        "rest_sample_size": (
            f"REST has only {len(feat_rest_all)} epochs total (2 short "
            f"recordings). The feature scaler is fit on REST; treat "
            f"scaling as fragile until more REST data exists."
        ),
    },
}

with open(output_dir / "preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ All outputs saved to {output_dir.resolve()}")

## Validation

The in-memory preprocessing outputs are validated against a series of structural and physiological checks: expected output files exist, epoch array shapes and channel counts are consistent, feature tables contain no NaN or infinite values, sampling frequencies match the expected per-domain values (REST 160 Hz, SLEEP 100 Hz, CHB 256 Hz), no `y_sleep` column exists anywhere (since no verified hypnogram is available), `y_seizure` is present only for CHB and is binary, and band powers, relative band powers, and entropy fall within their expected physiological/numerical ranges.

In [ ]:
print("\n" + "=" * 60)
print("STEP 13: VALIDATION & FINAL CHECKS")
print("=" * 60)

# ---- Output files exist ----
print("\nOutput files:")
for fname in ["features_rest.csv", "features_sleep.csv", "features_chb.csv",
              "features_rest.parquet", "features_sleep.parquet", "features_chb.parquet",
              "preprocessing_metadata.json"]:
    fpath = output_dir / fname
    status = "OK" if fpath.exists() else "MISSING"
    print(f"  [{status:7s}] {fname}")
    assert fpath.exists(), f"{fname} was not created"

# ---- Epoch arrays: shape consistency ----
print("\nEpoch array shapes:")
epoch_checks = [
    ("epochs_rest_r01.npy", ep_rest_r01.shape),
    ("epochs_rest_r02.npy", ep_rest_r02.shape),
    ("epochs_sleep.npy", ep_sleep.shape),
] + [(f"epochs_{rec}.npy", ep.shape) for rec, ep in ep_chb_dict.items()]

for fname, expected_shape in epoch_checks:
    actual = np.load(data_dir / fname)
    assert actual.shape == expected_shape, f"{fname}: shape mismatch"
    assert np.isfinite(actual).all(), f"{fname}: contains NaN/Inf"
    print(f"  [OK] {fname:24s} shape={actual.shape}")

# ---- Feature tables: row counts, NaN/Inf, required columns ----
print("\nFeature table checks:")
required_cols = {"recording_id", "epoch_idx", "t_start_sec", "t_center_sec", "t_end_sec"}

for name, df in [("REST", feat_rest_all), ("SLEEP", feat_sleep), ("CHB", feat_chb_all)]:
    n_nan = int(df.isna().sum().sum())
    n_inf = int(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())
    missing_cols = required_cols - set(df.columns)

    assert n_nan == 0, f"{name}: {n_nan} NaN values found"
    assert n_inf == 0, f"{name}: {n_inf} Inf values found"
    assert not missing_cols, f"{name}: missing columns {missing_cols}"

    print(f"  [OK] {name:6s} rows={len(df):5d}, cols={len(df.columns):2d}, "
          f"NaN={n_nan}, Inf={n_inf}")

# ---- Sampling frequency sanity (metadata-derived, not hardcoded) ----
assert np.isclose(sfreq_rest, 160.0), f"Unexpected REST sfreq: {sfreq_rest}"
assert np.isclose(sfreq_sleep, 100.0), f"Unexpected SLEEP sfreq: {sfreq_sleep}"
assert np.isclose(sfreq_chb, 256.0), f"Unexpected CHB sfreq: {sfreq_chb}"

# ---- No sleep-stage labels exist: explicit negative assertion ----
assert "y_sleep" not in feat_sleep.columns, (
    "y_sleep column found but no verified hypnogram source exists — "
    "this would be a fabricated label."
)

# ---- CHB seizure label checks ----
print("\nCHB seizure label checks:")
assert "y_seizure" in feat_chb_all.columns, "CHB feature table is missing y_seizure"
assert set(feat_chb_all["y_seizure"].unique()) <= {0, 1}, "y_seizure must contain only {0, 1}"
assert int(feat_chb_all["y_seizure"].sum()) == n_seizure_positive, (
    "y_seizure positive count drifted from the labeling step"
)
# REST/SLEEP must NOT silently acquire a seizure label of any kind —
# absence of annotation source must stay structurally absent, not 0.
assert "y_seizure" not in feat_rest_all.columns, "REST must not carry a y_seizure column"
assert "y_seizure" not in feat_sleep.columns, "SLEEP must not carry a y_seizure column"
print(f"  [OK] y_seizure in {{0,1}}, {n_seizure_positive} positive / {n_seizure_negative} negative")
print(f"  [OK] REST and SLEEP correctly have no y_seizure column")

# ---- Physiological / numerical sanity checks on raw (pre-scaling) features ----
print("\nFeature sanity checks (pre-scaling):")
band_cols = ["delta", "theta", "alpha", "beta"]
rel_cols = ["rel_delta", "rel_theta", "rel_alpha", "rel_beta"]
nonneg_cols = ["std", "var", "rms"] + band_cols

for name, df in [("REST", feat_rest_all), ("SLEEP", feat_sleep), ("CHB", feat_chb_all)]:
    for col in nonneg_cols:
        assert np.isfinite(df[col]).all(), f"{name}.{col}: non-finite values"
        assert (df[col] >= 0).all(), f"{name}.{col}: negative band power/amplitude found"

    for col in rel_cols:
        assert np.isfinite(df[col]).all(), f"{name}.{col}: non-finite values"
        assert (df[col] >= -1e-9).all() and (df[col] <= 1 + 1e-9).all(), (
            f"{name}.{col}: relative band power outside [0,1]"
        )

    assert np.isfinite(df["entropy"]).all(), f"{name}.entropy: non-finite values"

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            assert not df[col].isna().all(), f"{name}.{col}: entirely NaN"
            assert np.isfinite(df[col]).any(), f"{name}.{col}: entirely non-finite"

    scaled_cols = [c for c in df.columns if c.endswith("_scaled")]
    for col in scaled_cols:
        assert np.isfinite(df[col]).all(), f"{name}.{col}: non-finite scaled value"

    print(f"  [OK] {name:6s} band powers >= 0, relative powers in [0,1], "
          f"entropy finite, {len(scaled_cols)} scaled cols finite")

print("\n✓ All validation checks passed")

### Cold Read-Back Validation

The feature tables and epoch arrays actually written to disk are reloaded independently of any in-memory state and re-checked against the same structural expectations, to catch serialization issues (e.g. dtype coercion or truncated writes) that would not be visible from the in-memory objects alone. Reloaded CHB positive epochs are additionally confirmed to intersect the annotated seizure interval for `chb_s01_r03`.

In [ ]:
print("\n" + "=" * 60)
print("STEP 14: COLD READ-BACK VALIDATION (saved artifacts only)")
print("=" * 60)

# Everything above validates in-memory objects. This step re-reads only
# what was actually persisted to disk, independent of any in-memory
# state, to catch silent write/serialization bugs (e.g. dtype coercion
# on save, truncated writes).

print("\nReloading feature tables from Parquet...")
reloaded = {
    "REST": pd.read_parquet(output_dir / "features_rest.parquet"),
    "SLEEP": pd.read_parquet(output_dir / "features_sleep.parquet"),
    "CHB": pd.read_parquet(output_dir / "features_chb.parquet"),
}

expected_rows = {"REST": len(feat_rest_all), "SLEEP": len(feat_sleep), "CHB": len(feat_chb_all)}

for name, df in reloaded.items():
    assert len(df) == expected_rows[name], (
        f"{name}: reloaded row count {len(df)} != expected {expected_rows[name]}"
    )
    missing = required_cols - set(df.columns)
    assert not missing, f"{name}: reloaded table missing columns {missing}"
    n_nan = int(df.isna().sum().sum())
    n_inf = int(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())
    assert n_nan == 0, f"{name}: reloaded table has {n_nan} NaN values"
    assert n_inf == 0, f"{name}: reloaded table has {n_inf} Inf values"
    assert df["recording_id"].notna().all(), f"{name}: recording_id not preserved on reload"
    print(f"  [OK] {name:6s} reloaded rows={len(df):5d}, cols={len(df.columns):2d}, NaN=0, Inf=0")

# CHB-specific label checks on the reloaded (disk) copy
chb_reloaded = reloaded["CHB"]
assert "y_seizure" in chb_reloaded.columns, "Reloaded CHB parquet is missing y_seizure"
assert set(chb_reloaded["y_seizure"].unique()) <= {0, 1}, "Reloaded y_seizure not binary"

reloaded_positive = chb_reloaded.loc[chb_reloaded["y_seizure"] == 1]
assert len(reloaded_positive) == n_seizure_positive, (
    f"Reloaded positive-epoch count {len(reloaded_positive)} != {n_seizure_positive}"
)
# Every positive epoch on disk must genuinely intersect the annotated
# 2996-3036s interval for chb_s01_r03 (the only recording with a seizure).
seiz_start, seiz_end = expected_chb_intervals["chb_s01_r03"][0]
for _, row in reloaded_positive.iterrows():
    assert row["recording_id"] == "chb_s01_r03", (
        f"Unexpected positive epoch in {row['recording_id']}, only chb_s01_r03 has a seizure"
    )
    assert row["t_start_sec"] < seiz_end and row["t_end_sec"] > seiz_start, (
        f"Positive epoch [{row['t_start_sec']}, {row['t_end_sec']}) does not "
        f"actually intersect annotated interval [{seiz_start}, {seiz_end})"
    )
print(f"  [OK] CHB: {len(reloaded_positive)} reloaded y_seizure=1 epochs all intersect "
      f"[{seiz_start},{seiz_end})s in chb_s01_r03")

# REST/SLEEP must not have acquired a seizure label anywhere in the saved files
assert "y_seizure" not in reloaded["REST"].columns, "Reloaded REST parquet unexpectedly has y_seizure"
assert "y_seizure" not in reloaded["SLEEP"].columns, "Reloaded SLEEP parquet unexpectedly has y_seizure"
print("  [OK] REST/SLEEP reloaded tables correctly have no y_seizure column")

# ---- Reload epoch arrays with mmap_mode="r" (forces a real disk read) ----
print("\nReloading epoch arrays (mmap_mode='r')...")
mmap_checks = [
    ("epochs_rest_r01.npy", ep_rest_r01.shape),
    ("epochs_rest_r02.npy", ep_rest_r02.shape),
    ("epochs_sleep.npy", ep_sleep.shape),
] + [(f"epochs_{rec}.npy", ep.shape) for rec, ep in ep_chb_dict.items()]

for fname, expected_shape in mmap_checks:
    arr = np.load(data_dir / fname, mmap_mode="r")
    assert arr.shape == expected_shape, f"{fname}: mmap shape {arr.shape} != {expected_shape}"
    assert np.isfinite(np.asarray(arr)).all(), f"{fname}: mmap read contains NaN/Inf"
    print(f"  [OK] {fname:24s} n_epochs={arr.shape[0]:5d}, channels={arr.shape[1]:2d}, "
          f"samples/epoch={arr.shape[2]:5d}, finite=True")

# ---- Metadata JSON sanity ----
print("\nReloading preprocessing_metadata.json...")
with open(output_dir / "preprocessing_metadata.json") as f:
    saved_metadata = json.load(f)

assert saved_metadata["supervised_targets"]["y_seizure"]["status"] == "IMPLEMENTED"
assert saved_metadata["supervised_targets"]["y_seizure"]["n_positive_epochs"] == n_seizure_positive
assert "NOT AVAILABLE" in saved_metadata["supervised_targets"]["y_sleep"]
print("  [OK] metadata reflects y_seizure=IMPLEMENTED and y_sleep=NOT AVAILABLE")

print("\n✓ Cold read-back validation passed — saved artifacts match in-memory results")

## Pipeline Summary

The following cell prints a consolidated summary of the datasets processed, the preprocessing pipeline applied, the generated output artifacts, the supervised targets available (`y_seizure` for CHB; `y_sleep` unavailable), and the limitations carried forward into downstream modeling notebooks.

In [ ]:
print("\n" + "=" * 60)
print("PREPROCESSING V1 — SUMMARY")
print("=" * 60)

print(f"""
Datasets processed (each kept as a separate domain, never mixed):

  REST   2 recordings, {len(feat_rest_all):4d} epochs, {sfreq_rest:.0f} Hz, 64-channel montage
  SLEEP  1 recording,  {len(feat_sleep):4d} epochs, {sfreq_sleep:.0f} Hz, 2-channel montage (Fpz-Cz, Pz-Oz)
  CHB    3 recordings, {len(feat_chb_all):4d} epochs, {sfreq_chb:.0f} Hz, 23-channel montage
         y_seizure: {n_seizure_positive} positive / {n_seizure_negative} negative epochs
         (chb01_03.edf: 1 annotated seizure, 2996-3036s, parsed from chb01-summary.txt)

Pipeline applied to every recording:
  raw EDF -> channel type correction (Sleep) -> EEG-only selection ->
  {FILTER_LOW_HZ}-{FILTER_HIGH_HZ}Hz band-pass filter -> {EPOCH_DURATION_SEC:.0f}s / {EPOCH_OVERLAP:.0%} overlap epoching ->
  Welch-PSD band-power features -> stationarity diagnostics ->
  temporal metadata -> CHB seizure epoch labeling ->
  RobustScaler (fit on REST) -> save -> validation -> cold read-back validation

Outputs (outputs/):
  features_{{rest,sleep,chb}}.csv / .parquet  — one row per epoch (CHB includes y_seizure)
  preprocessed_data/epochs_*.npy              — filtered (channels, epoch) arrays
  preprocessing_metadata.json                 — params, sfreqs, recording IDs, seizure provenance

Cross-dataset evaluation readiness:
  - recording_id preserved on every row -> supports leave-one-dataset-out
    or leave-one-recording-out splitting without re-deriving identity.
  - No random epoch-level mixing was performed anywhere in this notebook.
  - Chronological order is preserved within each recording.

Supervised targets:
  - y_seizure (CHB only): IMPLEMENTED. Parsed from chb01-summary.txt with
    provenance retained in preprocessing_metadata.json. Interval-
    intersection labeling (not epoch-center), so an epoch is positive if
    it overlaps the annotated interval at all. REST and SLEEP structurally
    have no y_seizure column — their absence of a seizure annotation
    source is not represented as a negative label.
  - y_sleep: NOT AVAILABLE. No verified sleep-stage hypnogram exists in
    this repository. Sleep data is preprocessed and feature-extracted,
    but carries no supervised target. Guarded by an explicit assertion.

NOT implemented in this notebook (explicit boundaries, not oversights):
  - Forecasting/state index (NSDI-style scalar): project-state.md marks
    this mathematically undefined and OPEN. The old notebook's weighted
    index (0.5*std + 0.3*entropy + 0.2*theta) was an arbitrary, unjustified
    placeholder and has been removed rather than carried forward. The
    epoch_idx / t_*_sec columns are the only "index" this notebook
    produces, and they're a time axis, not a state/deviation measure.
  - No CNN/TCN/state-geometry/transition-prediction work, per the
    project's current-milestone boundary.

Known limitations carried into the modeling stage (not fixed here):
  - REST has only 4 epochs total; the feature scaler is fit on this small
    sample, so scaling should be treated as fragile until more REST data
    exists (see Step 11 comment and preprocessing_metadata.json).
  - CHB seizure labels come from a single subject/session (chb01); do not
    generalize positive/negative balance beyond this specific recording.
""")

print("=" * 60)